# Text-only Executor — can the scene graph alone drive the task?

This notebook tests a hypothesis from [`ORCHESTRATOR.md`](../ORCHESTRATOR.md): the LLM can plan correctly using **only the symbolic scene graph**, with no screenshot input on any step. The executor's system prompt already contains the SCENE GRAPH block (objects with ids, types, labels, bboxes; edges with anchored endpoints) and the ACTIVE SELECTION block (handle directions for the selected shape). The screenshot the executor normally gets is *extra* information on top of that.

**API**: `executor.infer()` now takes `screenshot_path` as an optional argument. Pass a path for the default screenshot+SG mode (used by `scene_graph_demo.ipynb` and `complex_tasks_demo.ipynb`); pass `None` for the text-only mode used here. The system prompt's `# INPUTS YOU RECEIVE` block adapts automatically — see `core/agents/executor.py`.

**Test case**: the same goal as `scene_graph_demo.ipynb` cell 25 — two labelled rectangles connected by an edge. With screenshots, the LLM finished in 7 steps (6 dispatched + 1 task_complete). The question: does it still succeed without them?

**Why this matters**

If the answer is *yes*, the orchestrator design's cost case lands: a saved compound tool can be replayed for free, and even fresh planning costs one LLM call per *plan*, not per *step*. Screenshots are then only needed at checkpoint failures — a tiny fraction of total operations.

**What we keep vs. what we drop**

| | Default `infer(...)` | `infer(..., screenshot_path=None)` |
|---|---|---|
| System prompt (catalog + SCENE GRAPH + active selection) | yes | yes |
| `# INPUTS YOU RECEIVE` block | mentions screenshot + SG | mentions SG only |
| User-message screenshot | yes | **no** |
| History of prior turns | yes | yes |
| Framework's internal CV scan after geometry ops | yes | yes |

Note: the framework still takes its own screenshots internally for handle detection (`_scan_and_reconcile`). We're only stripping the screenshot that the *LLM* sees.

In [1]:
# --- 0. Setup ---------------------------------------------------------------
import os, sys, time, json
sys.path.insert(0, os.path.abspath('..'))

import pyautogui
from core import config
from core.tools import dispatch, TOOL_CATALOG
from core.state import scene_graph as sg
from core.agents.executor import infer

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(name)s: %(message)s')

print(f'Loaded {len(TOOL_CATALOG)} tools, model = {config.llm_model()}.')

# Shared ui_graph (threaded through every dispatch so scene_graph + handles persist)
G = config.ui_graph()
G['scene_graph'] = sg.load()

Loaded 33 tools, model = qwen3.5:35b.


## How text-only inference works

Calling `infer(task, ui_graph, screenshot_path=None, history=...)` flips two things in the request, while keeping everything else identical to the screenshot variant:

1. **The user message has no `images=[...]` attached.** The LLM only sees the text `Task: …`.
2. **The system prompt's `# INPUTS YOU RECEIVE` block** is swapped to the *text-only* version — it tells the LLM the SCENE GRAPH is its sole view of the canvas, and to call `scan_handles` / `click_empty_canvas` rather than guess if state is missing.

Same catalog, same decision procedure, same scene-graph block — so any difference in behaviour between the two notebooks is attributable to the missing visual.

In [4]:
# --- 1. Focus + clean canvas + reset scene_graph ----------------------------
sg.reset()
G['scene_graph'] = sg.load()
G['selected_handles'] = None

print('Switch to draw.io NOW.')
for i in range(5, 0, -1):
    print(f'  {i}s …')
    time.sleep(1)
print('GO — cleaning canvas …')
pyautogui.click(1100, 800); time.sleep(0.3)
for _ in range(40):
    pyautogui.hotkey('command', 'z'); time.sleep(0.04)
pyautogui.click(1100, 800); time.sleep(0.3)
print('clean. scene_graph reset. Ready for text-only LLM.')

Switch to draw.io NOW.
  5s …
  4s …
  3s …
  2s …
  1s …
GO — cleaning canvas …
clean. scene_graph reset. Ready for text-only LLM.


In [5]:
# --- 2. Run the text-only loop ----------------------------------------------
# Same TASK and outer loop shape as scene_graph_demo.ipynb cell 25, except
# we call infer(...) with screenshot_path=None.  Every other bit of
# bookkeeping (history accumulation, dispatch, scene-graph printing) is
# identical so the two runs are directly comparable.

TASK = (
    "Place two rectangles on the canvas. Label the first 'Source' and "
    "the second 'Target'. Then draw a connector edge from Source to "
    "Target. Use one tool per step. When the canvas matches this "
    "description, emit task_complete."
)

MAX_STEPS = 12
history = []
trace = []
t_start = time.perf_counter()

for step in range(1, MAX_STEPS + 1):
    print(f'\n========== Text-only step {step} ==========')
    decision = infer(TASK, G, screenshot_path=None,
                     history=history if history else None)
    tool = decision.get('tool')
    params = decision.get('params', {}) or {}
    reasoning = decision.get('reasoning', '')
    print(f'reasoning: {reasoning}')
    print(f'tool:      {tool}({params})')

    if tool == 'task_complete':
        print('\U0001f3c1 LLM signalled task_complete')
        break
    if tool == 'request_rescan':
        print('… rescan requested')
        dispatch('scan_handles', {}, ui_graph=G)
        history.append({'role': 'assistant', 'content': json.dumps(decision)})
        history.append({'role': 'user', 'content': 'rescanned, what next?'})
        continue
    if tool not in TOOL_CATALOG:
        print(f'❌ unknown tool {tool}; aborting')
        break

    result = dispatch(tool, params, ui_graph=G)
    status = result.get('status')
    print(f'result:    status={status}'
          + (f', error={result.get("error")}' if status != 'ok' else ''))

    trace.append({'tool': tool, 'params': dict(params), 'result': result})
    history.append({'role': 'assistant',
                    'content': json.dumps({'tool': tool, 'params': params,
                                           'reasoning': reasoning})})
    history.append({'role': 'user',
                    'content': (f"Tool '{tool}' executed (status={status}). "
                                "What's the next step? Use 'task_complete' "
                                "if the goal is met.")})

    print('--- scene_graph after this step ---')
    print(sg.summary_for_prompt(G['scene_graph']))
    time.sleep(0.3)

t_total = time.perf_counter() - t_start
print('\n=========== FINAL scene_graph (text-only) ===========')
print(sg.summary_for_prompt(G['scene_graph']))
print(f'\n\U0001f4e6 {len(trace)} dispatched steps  •  {t_total:.1f}s wall-clock')

INFO   core.agents.executor: Querying qwen3.5:35b (text-only) …



========== Text-only step 1 ==========


INFO   httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO   core.agents.executor: Decided: place_shape  {'tool_name': 'Rectangle_Tool'}


reasoning: SCENE GRAPH shows an empty canvas with no shapes. Task requires placing two rectangles, labeling them 'Source' and 'Target', and connecting them. Starting with the first rectangle using place_shape.
tool:      place_shape({'tool_name': 'Rectangle_Tool'})


INFO   domains.drawio.operations:   [L1] place_shape('Rectangle_Tool') → click (36, 322) + Enter


...

result:    status=ok
--- scene_graph after this step ---
**Objects (2):**
  - `obj_001` Rectangle "Source"  bbox=[890,515,84x42]  *SELECTED*
  - `obj_002` Rectangle "Target"  bbox=[708,515,84x42]

**Edges (1):**
  - `edge_001` `obj_001`.w → `obj_002`.e

_(scene_graph op #9, last op: connect_shapes)_


INFO   core.agents.executor: Querying qwen3.5:35b (text-only) …



========== Text-only step 7 ==========


INFO   httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO   core.agents.executor: Decided: task_complete  {}


reasoning: SCENE GRAPH confirms all task requirements are met: obj_001 is labeled 'Source', obj_002 is labeled 'Target', and edge_001 connects obj_001 to obj_002. All three task requirements are satisfied.
tool:      task_complete({})
🏁 LLM signalled task_complete

=========== FINAL scene_graph (text-only) ===========
**Objects (2):**
  - `obj_001` Rectangle "Source"  bbox=[890,515,84x42]  *SELECTED*
  - `obj_002` Rectangle "Target"  bbox=[708,515,84x42]

**Edges (1):**
  - `edge_001` `obj_001`.w → `obj_002`.e

_(scene_graph op #9, last op: connect_shapes)_

📦 6 dispatched steps  •  122.0s wall-clock


In [6]:
# --- 3. Structural verification + comparison metrics ------------------------
scene = G['scene_graph']
n_objs  = len(scene.get('objects', []))
n_edges = len(scene.get('edges', []))
labels  = sorted(o.get('label', '') for o in scene.get('objects', []))

print(f'Objects: {n_objs}  Edges: {n_edges}  Labels: {labels}')

expected_labels = {'Source', 'Target'}
success = n_objs == 2 and n_edges >= 1 and set(labels) == expected_labels

# Tool-call distribution — useful for comparing against the screenshot run
from collections import Counter
call_dist = Counter(s['tool'] for s in trace)

print()
if success:
    print('\U0001f389 Text-only run matched the expected canvas '
          '(2 labelled rects + 1 connector).')
else:
    print('\u26a0  Text-only run did NOT match the expected canvas.')

print(f'\nTool-call distribution:')
for tool_name, count in call_dist.most_common():
    print(f'  {count}× {tool_name}')

print('\nReference (screenshot run, scene_graph_demo.ipynb cell 25):')
print('  Steps dispatched: 6 (+ task_complete)')
print("  Calls: 2× place_shape, 2× type_label, 1× move_shape, 1× connect_shapes")
print(f'\nThis run (text-only):')
print(f'  Steps dispatched: {len(trace)}')
print(f'  Calls: {dict(call_dist)}')

Objects: 2  Edges: 1  Labels: ['Source', 'Target']

🎉 Text-only run matched the expected canvas (2 labelled rects + 1 connector).

Tool-call distribution:
  2× place_shape
  2× type_label
  1× move_shape
  1× connect_shapes

Reference (screenshot run, scene_graph_demo.ipynb cell 25):
  Steps dispatched: 6 (+ task_complete)
  Calls: 2× place_shape, 2× type_label, 1× move_shape, 1× connect_shapes

This run (text-only):
  Steps dispatched: 6
  Calls: {'place_shape': 2, 'type_label': 2, 'move_shape': 1, 'connect_shapes': 1}


## What to make of the results

Things to compare against the screenshot run in `scene_graph_demo.ipynb` cell 25:

1. **Step count.** The screenshot run finished in 6 dispatched + 1 `task_complete`. If text-only finishes in roughly the same envelope (±2 steps), the visual was redundant for *this* task. If it spikes (10–12 steps) or hits `MAX_STEPS`, the visual was carrying real information the scene graph alone doesn't surface.

2. **Tool-call distribution.** Same set of operands, in different order, is still a positive result — it shows the LLM is making sound planning decisions from the symbolic state. Wildly different operands (e.g. lots of `click_empty_canvas` retries, `extend_shape` instead of `connect_shapes`) suggests the LLM is groping without the picture.

3. **Reasoning text.** Each printed `reasoning:` line should cite SCENE GRAPH state explicitly ("SCENE GRAPH shows X; need Y; therefore Z"). If the text-only run still produces these citations cleanly, the model is genuinely reasoning over the symbolic state, not pattern-matching pixels.

## What this would mean for ORCHESTRATOR.md

A clean pass here directly validates Phase 1 of the orchestrator design — the planner can emit a full graph from text + scene-graph alone, and screenshots are only needed when a checkpoint fails. The cost case in the comparison table becomes concrete:

| Mode | Screenshots per run | LLM calls per run |
|---|---|---|
| `infer(..., screenshot_path=img)` (screenshot+SG every step) | ~7 | ~7 |
| `infer(..., screenshot_path=None)` (text-only, this notebook) | 0 from LLM · ~3 from framework CV | ~7 |
| Orchestrator planner (Phase 1) | 0 unless checkpoint fails | **1** |

## Repeatability

Local models drift run-to-run. If you want a stronger claim, re-run cells 3–4 a few times and aggregate the success rate — the orchestrator paper would quote `success_rate@N` rather than a single trial. A multi-trial loop is the obvious next addition once this single-trial version is debugged.